In [0]:
!pip install numpy==1.26.4 openai==1.69.0 langchain-community==0.4.1 PyPDF2==3.0.1

In [0]:
dbutils.library.restartPython()

In [0]:
import os
source_folder =  "../knowledge"
dbfs_target_folder = "/Volumes/workspace/default/data_prep/"
os.makedirs(source_folder,exist_ok=True)
dbutils.fs.mkdirs(dbfs_target_folder)
subfolders = ['docs','images']
for subfolder in subfolders:
  local_subfolder = os.path.join(source_folder,subfolder)
  dbfs_subfolder = f"{dbfs_target_folder}/{subfolder}"
  os.makedirs(local_subfolder,exist_ok=True)
  dbutils.fs.mkdirs(dbfs_subfolder)
  for filename in os.listdir(local_subfolder):
    src = os.path.join(local_subfolder,filename)
    dst = f"{dbfs_subfolder}/{filename}"
    if os.path.isfile(src) and os.path.getsize(src)>0:
      dbutils.fs.cp(f"file:{os.path.abspath(src)}",dst,recurse=False)

In [0]:
print(dbutils.fs.ls("/Volumes/workspace/default/data_prep/images"))
print("\n images:")
print(dbutils.fs.ls("/Volumes/workspace/default/data_prep/docs"))
print("\n docs:")
print(dbutils.fs.ls("/Volumes/workspace/default/data_prep/"))


      

In [0]:
%sql
create catalog udemydatabricks

In [0]:
%sql
create schema if not exists udemydatabricks.RAG

In [0]:
images_df = spark.read.format("binaryFile").load("/Volumes/workspace/default/data_prep/images")
images_df.createOrReplaceTempView("udemydatabricks.RAG.images_temp")
spark.sql("""
          CREATE OR REPLACE table udemydatabricks.RAG.images_metadata AS
          select 
          path as content_path,
          base64(content) as base64_content
          from images_temp
          """)
display(spark.table("udemydatabricks.RAG.images_metadata"))

In [0]:
spark.sql("""
          create or REPLACE TABLE udemydatabricks.RAG.images_verbalisation as
          SELECT *, ai_query(
              'databricks-llama-4-maverick',
              'what is this image text about', files => unbase64(base64_content)
          )as chunk
          from udemydatabricks.RAG.images_metadata
          """)
display(spark.table("udemydatabricks.RAG.images_verbalisation"))


In [0]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from PyPDF2 import PdfReader

def perform_fixed_size_chunking(document, chunk_size=2000, chunk_overlap=500):
    text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    length_function = len,
    separators = ["\n\n",  "\n", ". ", " ",""])
    
    return text_splitter.split_text(document)
import os
docs_folder = "./knowledge/docs"
dbfs_docs_folder = "/Volumes/workspace/default/data_prep/docs"
all_docs = []

for filename in os.listdir(docs_folder):
  filepath = os.path.join(docs_folder,filename)
  dbfs_path = f"{dbfs_docs_folder}/{filename}"  
  if os.path.isfile(filepath) and filename.lower().endswith(".pdf"):
    with open(filepath,'rb') as f:
      reader = PdfReader(f)
      text=""
      for page in reader.pages:
          page_text = page.extract_text()
          if page_text:
              text += page_text
      if text.strip():
          chunks = perform_fixed_size_chunking(text)
          for i,chunk in enumerate(chunks):
              if chunk.strip():
                  all_docs.append({
                      "content_path":dbfs_path,
                      "chunk":chunk
                  })
if all_docs:
    df=spark.createDataFrame(all_docs)
    print(f"Total chunks created :{df.count()}")
    print(f"\n chunks per document")
    df.groupBy("content_path").count().show(truncate=False)
    display(df)
else:
    print("No chunks extrated from document")

df.write.mode("overwrite").saveAsTable("udemydatabricks.RAG.docs_chunks")



In [0]:
spark.sql("""
          CREATE or REPLACE TABLE udemydatabricks.RAG.final_rag_dataset AS
          SELECT monotonically_increasing_id() as id, content_path, chunk FROM
          udemydatabricks.RAG.images_verbalisation
          UNION ALL
          SELECT monotonically_increasing_id() as id, content_path, chunk FROM
          udemydatabricks.RAG.docs_chunks   
          """)
display(spark.table("udemydatabricks.RAG.final_rag_dataset"))
